# Analysis Report 03 — Full CZ→HCR Transform Characterisation

Comprehensive characterisation of the transform between CZ (in-vivo 2P) and HCR (ex-vivo lightsheet)
using ground-truth BigWarp landmark pairs from all available manually co-registered subjects.

## Subjects covered
| Subject | Date | Notes |
|---------|------|-------|
| 755252 | 2024-12-19 | |
| 767018 | — | No coreg dir available |
| 767022 | 2025-03-06 | |
| 782149 | 2025-05-01 | Thin lightsheet |
| 788406 | 2025-07-18 | |
| 790322 | 2025-07-10 | |

## Sections
1. Rotation & tilts  
2. Physical expansion rates (XY and Z)  
3. Translation & centre positions  
4. Tissue geometry & thickness  
5. Match rates (CZ matched / GFP+ / total HCR)  
6. Non-rigid deformation (rigid residual, per-cell distances)  
7. Z depth relationship (hcr\_z = f(cz\_z))  
8. XY deformation vs depth  
9. Spatial landmark coverage  
10. Summary table

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from scipy import stats

CODE_DIR = Path("/root/capsule/code")
DATA_DIR = Path("/root/capsule/data")
SCRATCH  = Path("/scratch")
sys.path.insert(0, str(CODE_DIR))

from coreg_data_loading import (
    load_landmarks, load_czstack_centroids,
    load_hcr_centroids, load_hcr_scales,
)
from coreg_alignment import rotation_matrix_euler

# ── Pixel resolutions ────────────────────────────────────────────────────────
# CZ: stored in BigWarp as pixels; physical = px × res
CZ_RES_Z  = 1.0    # µm/px
CZ_RES_XY = 0.78   # µm/px

# HCR: BigWarp used zarr pyramid level ≈ 1 µm/px for all dimensions
HCR_BW_RES = 1.0   # µm/px  (BigWarp-stored HCR coords)

GFP_COUNT_THRESHOLD = 5   # spot counts > threshold → GFP+

# ── Best landmark file suffix per subject ───────────────────────────────────
LM_KEYS = {
    "755252": "iter3_reordered_qced",
    "767022": "iter3_reordered_qced",
    "782149": "iter4_qced",
    "788406": "iter4_qced",
    "790322": "iter6_qced",
}
SUBJECTS = list(LM_KEYS.keys())
print(f"Subjects: {SUBJECTS}")

In [ ]:
# ── Load all data into per-subject dicts ─────────────────────────────────────
calib = pd.read_csv(DATA_DIR / "calibration_results.csv").set_index("subject")

data = {}  # subj → dict with lm_df, cz_df, hcr_df, sc_df, fp, scales

for subj in SUBJECTS:
    coreg_dirs = sorted(DATA_DIR.glob(f"{subj}_*_ctl-czstack-hcr-coreg_*"))
    if not coreg_dirs:
        print(f"{subj}: SKIPPED — no coreg dir")
        continue
    d = coreg_dirs[-1]

    # Landmark CSV
    key = LM_KEYS[subj]
    lm_csv = sorted(d.glob(f"*{key}*.csv"))
    lm_df  = load_landmarks(lm_csv[-1])
    active = lm_df[lm_df["active"]].copy()

    # Convert to physical µm (BigWarp pixel → µm)
    active["cz_z_um"] = active["czstack_z"] * CZ_RES_Z
    active["cz_y_um"] = active["czstack_y"] * CZ_RES_XY
    active["cz_x_um"] = active["czstack_x"] * CZ_RES_XY
    active["hcr_z_um"] = active["hcr_z"] * HCR_BW_RES
    active["hcr_y_um"] = active["hcr_y"] * HCR_BW_RES
    active["hcr_x_um"] = active["hcr_x"] * HCR_BW_RES

    # Apply known rotation to CZ
    r = calib.loc[int(subj)]
    R = rotation_matrix_euler(r["euler_z_deg"], r["euler_x_deg"], r["euler_y_deg"])
    cz_zyx = active[["cz_z_um", "cz_y_um", "cz_x_um"]].values
    cz_rot_zyx = (R @ cz_zyx.T).T
    active["cz_rot_z"] = cz_rot_zyx[:, 0]
    active["cz_rot_y"] = cz_rot_zyx[:, 1]
    active["cz_rot_x"] = cz_rot_zyx[:, 2]

    # CZ cell centroids (full population)
    cz_cent_csv = sorted(d.glob(f"*czstack*centroids*.csv"))
    cz_df = pd.read_csv(cz_cent_csv[0]) if cz_cent_csv else None

    # Filepaths JSON → HCR data
    fp_json = sorted(d.glob("*filepaths*.json"))[-1]
    fp = json.load(open(fp_json))
    hcr_scales = load_hcr_scales(fp["fused_json_file"])
    hcr_df = load_hcr_centroids(fp["hcr_centroid_path"])

    # Spot counts
    sc_csvs = sorted(d.glob("*spot*488*.csv"))
    sc_df   = pd.read_csv(sc_csvs[0]) if sc_csvs else None

    data[subj] = dict(
        lm=active, cz=cz_df, hcr=hcr_df, sc=sc_df,
        fp=fp, scales=hcr_scales, R=R, calib=r,
        coreg_dir=d,
    )
    n_sc = len(sc_df) if sc_df is not None else 0
    print(f"{subj}: {len(active)} landmarks  {len(cz_df) if cz_df is not None else '?'} CZ cells  "
          f"{len(hcr_df)} HCR cells  {n_sc} spot-count rows")

---
## 1. Rotation & Tilts
The main rotation axis is **euler_x** (≈ ±173–179°) in the convention  
`R = Ry(euler_y) @ Rx(euler_x) @ Rz(euler_z)`.  
This is a proper rotation (det R = +1). euler_z and euler_y are small tilts.

In [ ]:
rot_df = calib.loc[[int(s) for s in SUBJECTS],
                    ["euler_x_deg", "euler_z_deg", "euler_y_deg",
                     "rotation_angle_deg", "n_lm"]].copy()
rot_df.index = SUBJECTS

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Rotation angles across subjects", fontsize=13)

labels = SUBJECTS
x = np.arange(len(labels))

ax = axes[0]
ax.bar(x, rot_df["euler_x_deg"], color="steelblue")
ax.axhline(rot_df["euler_x_deg"].mean(), ls="--", color="navy",
           label=f'mean={rot_df["euler_x_deg"].mean():.1f}°')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30)
ax.set_ylabel("degrees")
ax.set_title("euler_x (main ~180° rotation)")
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(x, rot_df["euler_z_deg"], color="tomato")
ax.axhline(0, ls="--", color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30)
ax.set_title("euler_z (Z-tilt)")
ax.set_ylabel("degrees")

ax = axes[2]
ax.bar(x, rot_df["euler_y_deg"], color="seagreen")
ax.axhline(0, ls="--", color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30)
ax.set_title("euler_y (Y-tilt / roll)")
ax.set_ylabel("degrees")

plt.tight_layout()
plt.savefig(SCRATCH / "transform_01_rotation.png", dpi=150)
plt.show()

print("\nRotation summary:")
print(rot_df[["euler_x_deg","euler_z_deg","euler_y_deg","rotation_angle_deg"]]
      .describe().round(2))

---
## 2. Physical Expansion Rates (XY and Z)
Computed as RMS spread ratio:  
`scale = sqrt(Σ hcr_c²) / sqrt(Σ cz_rot_c²)` after centering and applying the known rotation.

HCR BigWarp coordinates use the zarr pyramid at **≈ 1 µm/px** for all dimensions.

In [ ]:
exp_rows = []
for subj, d in data.items():
    lm = d["lm"]
    cz_c  = lm[["cz_rot_z","cz_rot_y","cz_rot_x"]].values
    hcr_c = lm[["hcr_z_um","hcr_y_um","hcr_x_um"]].values
    cz_c  -= cz_c.mean(0)
    hcr_c -= hcr_c.mean(0)

    def rms(a): return np.sqrt((a**2).sum())

    sx = hcr_c[:,2].std() / cz_c[:,2].std()
    sy = hcr_c[:,1].std() / cz_c[:,1].std()
    sz = hcr_c[:,0].std() / cz_c[:,0].std()
    s_xy = rms(hcr_c[:,1:]) / rms(cz_c[:,1:])
    exp_rows.append(dict(subject=subj, n_lm=len(lm),
                         x_scale=sx, y_scale=sy, z_scale=sz,
                         xy_scale=s_xy, z_over_xy=sz/s_xy))

exp_df = pd.DataFrame(exp_rows).set_index("subject")
print(exp_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle("Physical expansion rates CZ→HCR", fontsize=13)

x = np.arange(len(exp_df))
w = 0.25
ax = axes[0]
ax.bar(x - w, exp_df["x_scale"], w, label="X", color="steelblue")
ax.bar(x,     exp_df["y_scale"], w, label="Y", color="cornflowerblue")
ax.bar(x + w, exp_df["z_scale"], w, label="Z", color="tomato")
ax.axhline(1, color="black", ls="--", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(exp_df.index, rotation=30)
ax.set_ylabel("Expansion rate (HCR / CZ)")
ax.set_title("Per-axis expansion")
ax.legend()

ax = axes[1]
ax.scatter(exp_df["xy_scale"], exp_df["z_scale"], s=80, zorder=3)
for subj, row in exp_df.iterrows():
    ax.annotate(subj, (row["xy_scale"], row["z_scale"]),
                textcoords="offset points", xytext=(5,3), fontsize=8)
lim = [1.0, 4.2]
ax.plot(lim, lim, "k--", lw=0.8, label="isotropic")
ax.set_xlabel("XY expansion rate"); ax.set_ylabel("Z expansion rate")
ax.set_title("Z vs XY expansion (isotropic=dashed)")
ax.legend()

plt.tight_layout()
plt.savefig(SCRATCH / "transform_02_expansion.png", dpi=150)
plt.show()

print(f"\nXY expansion: {exp_df.xy_scale.mean():.3f} ± {exp_df.xy_scale.std():.3f}  "
      f"range [{exp_df.xy_scale.min():.3f}, {exp_df.xy_scale.max():.3f}]")
print(f"Z  expansion: {exp_df.z_scale.mean():.3f} ± {exp_df.z_scale.std():.3f}  "
      f"range [{exp_df.z_scale.min():.3f}, {exp_df.z_scale.max():.3f}]")
print(f"Z/XY ratio:   {exp_df.z_over_xy.mean():.3f} ± {exp_df.z_over_xy.std():.3f}")

---
## 3. Translation & Centre Positions
t\_z, t\_y, t\_x from calibration_results.csv (translation that maps rotated CZ centroid → HCR centroid).  
Also compute the CZ centre in physical µm and where it lands in HCR space.

In [ ]:
trans_df = calib.loc[[int(s) for s in SUBJECTS],
                      ["t_z_um","t_y_um","t_x_um"]].copy()
trans_df.index = SUBJECTS

print("Translation (µm) per subject:")
print(trans_df.round(1))
print(f"\n  t_z: {trans_df.t_z_um.mean():.1f} ± {trans_df.t_z_um.std():.1f}  "
      f"range [{trans_df.t_z_um.min():.1f}, {trans_df.t_z_um.max():.1f}]")
print(f"  t_y: {trans_df.t_y_um.mean():.1f} ± {trans_df.t_y_um.std():.1f}  "
      f"range [{trans_df.t_y_um.min():.1f}, {trans_df.t_y_um.max():.1f}]")
print(f"  t_x: {trans_df.t_x_um.mean():.1f} ± {trans_df.t_x_um.std():.1f}  "
      f"range [{trans_df.t_x_um.min():.1f}, {trans_df.t_x_um.max():.1f}]")

print("\nCZ centre (µm) and corresponding HCR position per subject:")
print(f"{'subj':>8}  {'CZ ctr z':>10}  {'CZ ctr y':>10}  {'CZ ctr x':>10}  "
      f"{'HCR ctr z':>10}  {'HCR ctr y':>10}  {'HCR ctr x':>10}")
for subj, d in data.items():
    lm = d["lm"]
    cz_ctr  = lm[["cz_z_um","cz_y_um","cz_x_um"]].mean().values
    hcr_ctr = lm[["hcr_z_um","hcr_y_um","hcr_x_um"]].mean().values
    print(f"{subj:>8}  {cz_ctr[0]:>10.1f}  {cz_ctr[1]:>10.1f}  {cz_ctr[2]:>10.1f}  "
          f"{hcr_ctr[0]:>10.1f}  {hcr_ctr[1]:>10.1f}  {hcr_ctr[2]:>10.1f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Translation (calibration) per subject", fontsize=13)
for i, (col, label) in enumerate([("t_z_um","t_z (µm)"),("t_y_um","t_y (µm)"),("t_x_um","t_x (µm)")]):
    ax = axes[i]
    ax.bar(np.arange(len(SUBJECTS)), trans_df[col], color=["steelblue","tomato","seagreen"][i])
    ax.axhline(trans_df[col].mean(), ls="--", color="black", lw=0.8,
               label=f'mean={trans_df[col].mean():.0f} µm')
    ax.set_xticks(np.arange(len(SUBJECTS)))
    ax.set_xticklabels(SUBJECTS, rotation=30)
    ax.set_title(label); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(SCRATCH / "transform_03_translation.png", dpi=150)
plt.show()

---
## 4. Tissue Geometry & Thickness
Both CZ and HCR imaging volumes include blank regions where no cells appear:
- **CZ**: blank at the top (before the pia surface); may have blank at the bottom too
- **HCR**: blank above and below the tissue section

**Robust surface estimation** — immune to rare junk ROIs outside the tissue:
1. Keep only cells in the **center-half XY region** (eliminates fringe/edge artefacts)
2. Sort by z; take the **median of the `n_edge` most-extreme z values**
3. `n_edge = 5` for CZ (very few cells near the surface → small n needed)  
   `n_edge = 100` for HCR (many cells; 100-cell median is stable)

**Anchor fraction**: (median matched-landmark HCR z − HCR tissue surface z) / HCR tissue thickness  
→ where in the HCR z stack the CZ volume sits (0 = surface, 1 = deep edge)

In [ ]:
def robust_tissue_bounds(z, y, x, n_edge=50):
    """Robustly estimate where tissue starts and ends along z.

    Steps:
      1. Keep only cells in the center-half XY region (removes fringe/edge junk).
      2. Sort by z; take median of the n_edge shallowest and n_edge deepest cells.

    Returns (z_surface, z_deep, thickness_um). Robust to a handful of junk ROIs.
    """
    z = np.asarray(z, dtype=float)
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    y_mid = (y.max() + y.min()) / 2
    x_mid = (x.max() + x.min()) / 2
    y_qtr = (y.max() - y.min()) / 4
    x_qtr = (x.max() - x.min()) / 4
    mask  = (np.abs(y - y_mid) <= y_qtr) & (np.abs(x - x_mid) <= x_qtr)
    z_ctr = z[mask] if mask.sum() >= n_edge * 2 else z
    sorted_z  = np.sort(z_ctr)
    z_surface = float(np.median(sorted_z[:n_edge]))
    z_deep    = float(np.median(sorted_z[-n_edge:]))
    return z_surface, z_deep, z_deep - z_surface

In [ ]:
N_EDGE_CZ  = 5    # few junk ROIs near CZ surface; small n is sufficient
N_EDGE_HCR = 100  # HCR has many cells; median of 100 is stable

thick_rows = []

# ── Z histogram figure (blank regions) ───────────────────────────────────────
fig_h, axes_h = plt.subplots(len(data), 2, figsize=(10, 3 * len(data)))
fig_h.suptitle("Z cell density — blank regions at top/bottom", fontsize=13)

for i, (subj, d) in enumerate(data.items()):
    lm    = d["lm"]
    sc_hcr = d["scales"]

    # ── CZ tissue bounds ─────────────────────────────────────────────────────
    if d["cz"] is not None:
        cz_z_all = d["cz"]["czstack_z"].values * CZ_RES_Z
        cz_y_all = d["cz"]["czstack_y"].values * CZ_RES_XY
        cz_x_all = d["cz"]["czstack_x"].values * CZ_RES_XY
    else:
        cz_z_all = lm["cz_z_um"].values
        cz_y_all = lm["cz_y_um"].values
        cz_x_all = lm["cz_x_um"].values
    cz_z_surface, cz_z_deep, cz_thickness = robust_tissue_bounds(
        cz_z_all, cz_y_all, cz_x_all, n_edge=N_EDGE_CZ)

    # ── HCR tissue bounds ─────────────────────────────────────────────────────
    hcr_z_native = d["hcr"]["hcr_z"].values * sc_hcr["scale_z"]
    hcr_y_native = d["hcr"]["hcr_y"].values * sc_hcr["scale_y"]
    hcr_x_native = d["hcr"]["hcr_x"].values * sc_hcr["scale_x"]
    hcr_z_surface, hcr_z_deep, hcr_thickness = robust_tissue_bounds(
        hcr_z_native, hcr_y_native, hcr_x_native, n_edge=N_EDGE_HCR)

    # ── HCR matched z range ──────────────────────────────────────────────────
    hcr_lm_z = lm["hcr_z_um"].values
    hcr_lm_z_min, hcr_lm_z_max = hcr_lm_z.min(), hcr_lm_z.max()

    # Anchor fraction: where median matched HCR z sits in HCR tissue z range
    hcr_lm_median_z = np.median(hcr_lm_z)
    anchor_frac = (hcr_lm_median_z - hcr_z_surface) / hcr_thickness

    thick_rows.append(dict(
        subject=subj,
        cz_z_surface=cz_z_surface, cz_z_deep=cz_z_deep, cz_thickness=cz_thickness,
        hcr_z_surface=hcr_z_surface, hcr_z_deep=hcr_z_deep, hcr_thickness=hcr_thickness,
        hcr_lm_z_min=hcr_lm_z_min, hcr_lm_z_max=hcr_lm_z_max,
        hcr_lm_spread=hcr_lm_z_max - hcr_lm_z_min,
        hcr_anchor_frac=anchor_frac,
        hcr_anchor_z=hcr_lm_median_z,
    ))

    # ── z histograms ─────────────────────────────────────────────────────────
    ax = axes_h[i, 0]
    ax.hist(cz_z_all, bins=50, color="steelblue", alpha=0.7)
    ax.axvline(cz_z_surface, color="red",  lw=2, label=f"surface={cz_z_surface:.0f} µm")
    ax.axvline(cz_z_deep,    color="navy", lw=2, label=f"deep={cz_z_deep:.0f} µm")
    ax.set_xlabel("CZ z (µm)"); ax.set_ylabel("# cells")
    ax.set_title(f"{subj} CZ  (n_edge={N_EDGE_CZ})")
    ax.legend(fontsize=7)

    ax = axes_h[i, 1]
    ax.hist(hcr_z_native, bins=80, color="tomato", alpha=0.7)
    ax.axvline(hcr_z_surface, color="red",  lw=2, label=f"surface={hcr_z_surface:.0f} µm")
    ax.axvline(hcr_z_deep,    color="navy", lw=2, label=f"deep={hcr_z_deep:.0f} µm")
    ax.set_xlabel("HCR z (µm)"); ax.set_ylabel("# cells")
    ax.set_title(f"{subj} HCR  (n_edge={N_EDGE_HCR})")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(SCRATCH / "transform_04a_z_histograms.png", dpi=150)
plt.show()

thick_df = pd.DataFrame(thick_rows).set_index("subject")
print("Tissue geometry summary (µm) — robust surface estimation:")
print(thick_df.round(1))

# ── Thickness comparison plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Tissue geometry (z-axis, µm)", fontsize=13)
x = np.arange(len(thick_df))

ax = axes[0]
ax.bar(x, thick_df["cz_thickness"], color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(thick_df.index, rotation=30)
ax.set_title("CZ tissue thickness (µm)"); ax.set_ylabel("µm")

ax = axes[1]
ax.bar(x, thick_df["hcr_thickness"], color="tomato", label="HCR tissue (robust)")
ax.bar(x, thick_df["hcr_lm_spread"], color="orange", label="HCR matched z span", alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(thick_df.index, rotation=30)
ax.set_title("HCR z thickness (µm)"); ax.set_ylabel("µm")
ax.legend(fontsize=8)

ax = axes[2]
ax.bar(x, thick_df["hcr_anchor_frac"], color="seagreen")
ax.axhline(thick_df["hcr_anchor_frac"].mean(), ls="--", color="black", lw=0.8,
           label=f'mean={thick_df.hcr_anchor_frac.mean():.2f}')
ax.set_xticks(x); ax.set_xticklabels(thick_df.index, rotation=30)
ax.set_title("HCR anchor fraction (0=surface, 1=deep)"); ax.set_ylabel("fraction")
ax.set_ylim(0, 1); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(SCRATCH / "transform_04_thickness.png", dpi=150)
plt.show()

---
## 5. Match Rates
- **match_rate**: n\_landmarks (matched pairs) / n\_cz\_cells  
- **gfp_rate**: n\_gfp+ HCR cells (spot\_count > threshold) / n\_hcr\_total  
- **coverage**: n\_landmarks / n\_gfp+ HCR cells (how well matched)

In [ ]:
mr_rows = []
for subj, d in data.items():
    n_lm  = len(d["lm"])
    n_cz  = len(d["cz"]) if d["cz"] is not None else None
    n_hcr = len(d["hcr"])

    if d["sc"] is not None:
        n_gfp = (d["sc"]["counts"] > GFP_COUNT_THRESHOLD).sum()
    else:
        n_gfp = None

    match_rate  = n_lm / n_cz  if n_cz  else None
    gfp_rate    = n_gfp / n_hcr if n_gfp else None
    coverage    = n_lm / n_gfp  if n_gfp else None

    mr_rows.append(dict(subject=subj, n_lm=n_lm, n_cz=n_cz, n_hcr=n_hcr, n_gfp=n_gfp,
                        match_rate=match_rate, gfp_rate=gfp_rate,
                        lm_per_gfp=coverage))

mr_df = pd.DataFrame(mr_rows).set_index("subject")
print("Match rate summary:")
print(mr_df.round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Match rates across subjects", fontsize=13)
x = np.arange(len(mr_df))

ax = axes[0]
ax.bar(x, mr_df["match_rate"], color="steelblue")
if mr_df.match_rate.notna().any():
    ax.axhline(mr_df.match_rate.mean(), ls="--", color="navy",
               label=f'mean={mr_df.match_rate.mean():.2f}')
ax.set_xticks(x); ax.set_xticklabels(mr_df.index, rotation=30)
ax.set_title("Match rate (n_lm / n_cz)"); ax.set_ylabel("fraction"); ax.legend(fontsize=8)
ax.set_ylim(0, 1)

ax = axes[1]
n_vals = [mr_df.loc[s,"n_cz"] or 0 for s in mr_df.index]
ax.bar(x - 0.2, n_vals, 0.4, label="n_cz", color="steelblue")
ax.bar(x + 0.2, mr_df["n_lm"].values, 0.4, label="n_lm", color="tomato")
ax.set_xticks(x); ax.set_xticklabels(mr_df.index, rotation=30)
ax.set_title("CZ cells vs matched pairs"); ax.legend(fontsize=8)

ax = axes[2]
gfp_vals = mr_df["n_gfp"].fillna(0).astype(int).values
ax.bar(x - 0.2, gfp_vals, 0.4, label=f"n_gfp (>{GFP_COUNT_THRESHOLD})", color="seagreen")
ax.bar(x + 0.2, mr_df["n_lm"].values, 0.4, label="n_lm", color="tomato")
ax.set_xticks(x); ax.set_xticklabels(mr_df.index, rotation=30)
ax.set_title("GFP+ HCR cells vs matched pairs"); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(SCRATCH / "transform_05_matchrate.png", dpi=150)
plt.show()

---
## 6. Non-Rigid Deformation
- **Rigid residual** from calibration (median per-cell 3D distance after rigid fit)  
- **Per-cell distance distribution**: histogram of point-wise residuals after rigid alignment

In [ ]:
rigid_res = calib.loc[[int(s) for s in SUBJECTS], "rigid_residual_median_um"].copy()
rigid_res.index = SUBJECTS

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Non-rigid deformation (rigid fit residuals)", fontsize=13)

ax = axes[0]
ax.bar(np.arange(len(rigid_res)), rigid_res.values, color="tomato")
ax.axhline(rigid_res.mean(), ls="--", color="black",
           label=f'mean={rigid_res.mean():.0f} µm')
ax.set_xticks(np.arange(len(rigid_res)))
ax.set_xticklabels(SUBJECTS, rotation=30)
ax.set_title("Rigid fit median residual (µm)")
ax.set_ylabel("µm"); ax.legend(fontsize=8)

ax = axes[1]
colors = plt.cm.tab10(np.linspace(0, 1, len(SUBJECTS)))
for i, (subj, d) in enumerate(data.items()):
    lm = d["lm"]
    r  = d["calib"]
    t  = np.array([r["t_z_um"], r["t_y_um"], r["t_x_um"]])

    # Rigid-aligned CZ position
    cz_aligned = lm[["cz_rot_z","cz_rot_y","cz_rot_x"]].values + t
    hcr_pos    = lm[["hcr_z_um","hcr_y_um","hcr_x_um"]].values
    residuals  = np.linalg.norm(hcr_pos - cz_aligned, axis=1)

    ax.hist(residuals, bins=60, alpha=0.5, color=colors[i],
            label=f"{subj} (med={np.median(residuals):.0f}µm)")

ax.set_xlabel("3D residual after rigid alignment (µm)")
ax.set_ylabel("Count")
ax.set_title("Per-cell rigid residual distribution")
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(SCRATCH / "transform_06_nonrigid.png", dpi=150)
plt.show()
print(f"Rigid residual (median): {rigid_res.mean():.0f} ± {rigid_res.std():.0f} µm  "
      f"range [{rigid_res.min():.0f}, {rigid_res.max():.0f}] µm")

---
## 7. Z Depth Relationship: hcr\_z = f(cz\_z)
Fit `hcr_z = z_scale × cz_z + t_z` per subject after applying the rotation.  
The residual pattern reveals non-linear (non-rigid) Z deformation.

In [ ]:
fig, axes = plt.subplots(2, len(data), figsize=(4 * len(data), 8))
fig.suptitle("Z depth relationship per subject", fontsize=13)

zfit_rows = []
for i, (subj, d) in enumerate(data.items()):
    lm = d["lm"]
    cz_z  = lm["cz_rot_z"].values   # rotated CZ z (µm)
    hcr_z = lm["hcr_z_um"].values   # HCR z (µm)

    slope, intercept, r, p, _ = stats.linregress(cz_z, hcr_z)
    residuals = hcr_z - (slope * cz_z + intercept)

    zfit_rows.append(dict(subject=subj, z_scale_fit=slope,
                          t_z_fit=intercept, r2=r**2,
                          residual_std=residuals.std(),
                          residual_median_abs=np.median(np.abs(residuals))))

    # Scatter + fit
    ax = axes[0, i]
    ax.scatter(cz_z, hcr_z, s=2, alpha=0.4)
    zz = np.linspace(cz_z.min(), cz_z.max(), 100)
    ax.plot(zz, slope * zz + intercept, 'r-', lw=1.5,
            label=f'z_scale={slope:.2f}\nr²={r**2:.3f}')
    ax.set_xlabel("CZ z_rot (µm)"); ax.set_ylabel("HCR z (µm)")
    ax.set_title(subj); ax.legend(fontsize=7)

    # Residual vs depth
    ax = axes[1, i]
    ax.scatter(cz_z, residuals, s=2, alpha=0.4)
    ax.axhline(0, color='red', lw=0.8)
    ax.set_xlabel("CZ z_rot (µm)")
    ax.set_ylabel("z residual (µm)")
    ax.set_title(f"std={residuals.std():.1f} µm")

plt.tight_layout()
plt.savefig(SCRATCH / "transform_07_zdepth.png", dpi=150)
plt.show()

zfit_df = pd.DataFrame(zfit_rows).set_index("subject")
print("\nZ linear fit per subject:")
print(zfit_df.round(3))
print(f"\nz_scale:  {zfit_df.z_scale_fit.mean():.3f} ± {zfit_df.z_scale_fit.std():.3f}")
print(f"t_z fit:  {zfit_df.t_z_fit.mean():.1f} ± {zfit_df.t_z_fit.std():.1f} µm")
print(f"R²:       {zfit_df.r2.mean():.4f} ± {zfit_df.r2.std():.4f}")
print(f"residual std:  {zfit_df.residual_std.mean():.1f} ± {zfit_df.residual_std.std():.1f} µm")

---
## 8. XY Deformation vs Depth
After rigid alignment (R + t), compute the per-cell **XY residual** as a function of CZ depth (z\_rot).  
A depth-dependent XY pattern indicates non-rigid deformation in XY.

In [ ]:
fig, axes = plt.subplots(2, len(data), figsize=(4 * len(data), 8))
fig.suptitle("XY deformation vs depth per subject", fontsize=13)

for i, (subj, d) in enumerate(data.items()):
    lm = d["lm"]
    r  = d["calib"]
    t  = np.array([r["t_z_um"], r["t_y_um"], r["t_x_um"]])

    cz_aligned_yx = lm[["cz_rot_y","cz_rot_x"]].values + t[1:]
    hcr_yx        = lm[["hcr_y_um","hcr_x_um"]].values
    xy_resid      = hcr_yx - cz_aligned_yx          # (N,2)
    xy_resid_norm = np.linalg.norm(xy_resid, axis=1)  # (N,)
    cz_z = lm["cz_rot_z"].values

    # Scatter: XY residual magnitude vs depth
    ax = axes[0, i]
    ax.scatter(cz_z, xy_resid_norm, s=2, alpha=0.3)
    # Running median
    order = np.argsort(cz_z)
    w = max(20, len(cz_z) // 10)
    med_z, med_r = [], []
    for j in range(0, len(cz_z) - w, w // 2):
        idx = order[j:j+w]
        med_z.append(cz_z[order[j + w//2]])
        med_r.append(np.median(xy_resid_norm[idx]))
    ax.plot(med_z, med_r, 'r-', lw=2, label="running median")
    ax.set_xlabel("CZ z_rot (µm)"); ax.set_ylabel("|XY residual| (µm)")
    ax.set_title(f"{subj} XY residual vs depth")
    ax.legend(fontsize=7)

    # Vector field: mean XY offset in z-bins
    ax = axes[1, i]
    z_bins = np.linspace(cz_z.min(), cz_z.max(), 8)
    for j in range(len(z_bins) - 1):
        mask = (cz_z >= z_bins[j]) & (cz_z < z_bins[j+1])
        if mask.sum() < 5: continue
        mean_pos_y = lm["cz_rot_y"].values[mask].mean()
        mean_pos_x = lm["cz_rot_x"].values[mask].mean()
        mean_dy    = xy_resid[mask, 0].mean()
        mean_dx    = xy_resid[mask, 1].mean()
        z_mid      = (z_bins[j] + z_bins[j+1]) / 2
        ax.annotate("", xy=(mean_pos_x + mean_dx, mean_pos_y + mean_dy),
                    xytext=(mean_pos_x, mean_pos_y),
                    arrowprops=dict(arrowstyle="->", color=plt.cm.viridis(j / len(z_bins))))
        ax.text(mean_pos_x, mean_pos_y, f"z={z_mid:.0f}", fontsize=6, color="grey")
    ax.set_xlabel("CZ x_rot (µm)"); ax.set_ylabel("CZ y_rot (µm)")
    ax.set_title("Mean XY offset per depth bin")
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(SCRATCH / "transform_08_xydeform.png", dpi=150)
plt.show()

---
## 9. Spatial Landmark Coverage
Where in the CZ volume are matched landmarks located?  
Gaps indicate regions where auto-matching may be weaker.

In [ ]:
n_sub = len(data)
fig, axes = plt.subplots(n_sub, 3, figsize=(12, 3 * n_sub))
fig.suptitle("Spatial landmark coverage (CZ space)", fontsize=13)

for i, (subj, d) in enumerate(data.items()):
    lm = d["lm"]
    cz = d["cz"]

    cz_z = lm["cz_z_um"].values
    cz_y = lm["cz_y_um"].values
    cz_x = lm["cz_x_um"].values

    # Background: all CZ cells
    if cz is not None:
        all_z = cz["czstack_z"].values * CZ_RES_Z
        all_y = cz["czstack_y"].values * CZ_RES_XY
        all_x = cz["czstack_x"].values * CZ_RES_XY
    else:
        all_z = all_y = all_x = None

    for j, (xa, ya, xt, yt, title) in enumerate([
        (cz_x, cz_y, all_x, all_y, "XY (top view)"),
        (cz_x, cz_z, all_x, all_z, "XZ (side view)"),
        (cz_z, cz_y, all_z, all_y, "ZY (side view)"),
    ]):
        ax = axes[i, j]
        if xt is not None:
            ax.scatter(xt, yt, s=1, c="lightgrey", alpha=0.4, rasterized=True)
        ax.scatter(xa, ya, s=4, c=cz_z, cmap="viridis", alpha=0.7, rasterized=True)
        ax.set_xlabel(title.split("(")[0].split("/")[0].strip()[-1])
        ax.set_ylabel(title.split("(")[0].split("/")[-1].strip()[0])
        if j == 0:
            ax.set_ylabel(f"{subj}\n" + title.split("(")[0].split("/")[-1].strip()[0])
        ax.set_title(title if i == 0 else "")
        ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.savefig(SCRATCH / "transform_09_coverage.png", dpi=120)
plt.show()

---
## 10. Summary Table

In [ ]:
summary = pd.DataFrame(index=SUBJECTS)

# Rotation
summary["euler_x_deg"]  = [calib.loc[int(s), "euler_x_deg"]  for s in SUBJECTS]
summary["euler_z_deg"]  = [calib.loc[int(s), "euler_z_deg"]  for s in SUBJECTS]
summary["euler_y_deg"]  = [calib.loc[int(s), "euler_y_deg"]  for s in SUBJECTS]

# Expansion
for col in ["xy_scale", "z_scale", "z_over_xy"]:
    summary[col] = exp_df[col]

# Translation
for col in ["t_z_um", "t_y_um", "t_x_um"]:
    summary[col] = trans_df[col]

# Thickness
for col in ["cz_thickness", "hcr_thickness", "hcr_anchor_frac"]:
    summary[col] = thick_df[col]

# Match rate
for col in ["n_lm", "n_cz", "n_hcr", "n_gfp", "match_rate"]:
    summary[col] = mr_df[col]

# Rigid residual
summary["rigid_residual_um"] = rigid_res

# Z linear fit
for col in ["z_scale_fit", "t_z_fit", "r2", "residual_std"]:
    summary[col] = zfit_df[col]

print("=" * 60)
print("FULL TRANSFORM CHARACTERISATION SUMMARY")
print("=" * 60)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print(summary.round(3).T)

print("\n" + "=" * 60)
print("CROSS-SUBJECT STATISTICS")
print("=" * 60)
stats_cols = ["euler_x_deg","euler_z_deg","euler_y_deg",
               "xy_scale","z_scale","z_over_xy",
               "t_z_um","t_y_um","t_x_um",
               "cz_thickness","hcr_thickness","hcr_anchor_frac",
               "match_rate","rigid_residual_um","z_scale_fit","residual_std"]
print(summary[stats_cols].describe().round(3).T
      [["mean","std","min","max"]])

print("\n" + "=" * 60)
print("KEY FACTS FOR PIPELINE DESIGN")
print("=" * 60)
m = summary.mean()
s = summary.std()
print(f"  Rotation (euler_x):  {m.euler_x_deg:.1f} ± {s.euler_x_deg:.1f}°  "
      f"range [{summary.euler_x_deg.min():.1f}, {summary.euler_x_deg.max():.1f}]")
print(f"  Euler_z tilt:        {m.euler_z_deg:.1f} ± {s.euler_z_deg:.1f}°")
print(f"  Euler_y tilt:        {m.euler_y_deg:.1f} ± {s.euler_y_deg:.1f}°")
print(f"  XY expansion:        {m.xy_scale:.3f} ± {s.xy_scale:.3f}×")
print(f"  Z expansion:         {m.z_scale:.3f} ± {s.z_scale:.3f}×  (linear fit z_scale: {m.z_scale_fit:.3f})")
print(f"  Z/XY anisotropy:     {m.z_over_xy:.2f}×")
print(f"  Translation t_z:     {m.t_z_um:.0f} ± {s.t_z_um:.0f} µm")
print(f"  Translation t_y:     {m.t_y_um:.0f} ± {s.t_y_um:.0f} µm")
print(f"  Translation t_x:     {m.t_x_um:.0f} ± {s.t_x_um:.0f} µm")
print(f"  HCR anchor fraction: {m.hcr_anchor_frac:.2f} ± {s.hcr_anchor_frac:.2f}")
print(f"  CZ tissue depth:     {m.cz_thickness:.0f} ± {s.cz_thickness:.0f} µm")
print(f"  HCR tissue depth:    {m.hcr_thickness:.0f} ± {s.hcr_thickness:.0f} µm")
print(f"  Match rate:          {m.match_rate:.2f} ± {s.match_rate:.2f}  (n_lm/n_cz)")
print(f"  Rigid residual:      {m.rigid_residual_um:.0f} ± {s.rigid_residual_um:.0f} µm  (non-rigid is essential)")
print(f"  Z linear fit R²:     {m.r2:.4f}  (residual std: {m.residual_std:.1f} µm)")

In [ ]:
# ── Save full summary CSV ─────────────────────────────────────────────────────
out_csv = DATA_DIR / "transform_analysis_summary.csv"
summary.to_csv(out_csv)
print(f"Saved: {out_csv}")

# Also list all figures saved
for f in sorted(SCRATCH.glob("transform_0*.png")):
    print(f"  {f}")